# HSTU_CANONICAL_v1 — Finalizer PATCHED v2

Loads the canonical finalizer definitions from the repo, verifies `finalize()` exists, then overrides only checkpoint resolution with recursive Google Drive auto-discovery. No retraining.

In [ ]:
# 1) Load the canonical finalizer definitions from GitHub.
import json, urllib.request

RAW = "https://raw.githubusercontent.com/hanialshater/Sparsewalker-/main/experiments/hstu_reproduction/HSTU_CANONICAL_v1_Finalize_Colab.ipynb"
print("Loading canonical finalizer:", RAW)
with urllib.request.urlopen(RAW) as r:
    canonical_nb = json.load(r)

skip_markers = (
    "stamp, summary_df, latency_df = finalize()",
    'print("HSTU_CANONICAL_v1:", stamp["status"])',
    "latency_df.pivot_table",
)
executed = 0
for i, cell in enumerate(canonical_nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue
    src = "".join(cell.get("source", []))
    if not src.strip() or any(m in src for m in skip_markers):
        continue
    print(f"Executing canonical cell {i}...")
    exec(compile(src, f"canonical_cell_{i}", "exec"), globals(), globals())
    executed += 1

assert "finalize" in globals(), "Canonical finalizer failed to define finalize()."
assert "load_canonical_model" in globals(), "Canonical finalizer failed to define load_canonical_model()."
print("Canonical definitions loaded:", executed, "code cells; finalize() is available.")

In [ ]:
# 2) Patch checkpoint resolution only.
from pathlib import Path
import torch

RESOLVED_CHECKPOINTS = {}

def _checkpoint_quality(ck):
    metrics = ck.get("metrics", {}) or {}
    if "NDCG@10" in metrics:
        return float(metrics["NDCG@10"])
    history = ck.get("history", []) or []
    vals = [float(r["NDCG@10"]) for r in history if isinstance(r, dict) and "NDCG@10" in r]
    return max(vals) if vals else float("-inf")

def _checkpoint_variant(path, ck):
    cfg = ck.get("config", {}) or {}
    m = str(cfg.get("model", "")).lower()
    if m in ("core", "large"):
        return m
    p = str(path).lower()
    if "hstu_core" in p: return "core"
    if "hstu_large" in p: return "large"
    return None

def resolve_checkpoint(variant):
    if variant in RESOLVED_CHECKPOINTS:
        return RESOLVED_CHECKPOINTS[variant]

    mydrive = Path("/content/drive/MyDrive")
    expected = DRIVE_ROOT / f"hstu_{variant}_seed{SEED}"
    candidates = []

    # Exact expected folder first.
    for name in ("best.pt", "latest.pt", "last.pt"):
        p = expected / name
        if p.exists(): candidates.append(p)

    # Then recursively search all MyDrive.
    patterns = [
        f"**/hstu_{variant}_seed{SEED}/best.pt",
        f"**/hstu_{variant}_seed{SEED}/latest.pt",
        f"**/hstu_{variant}_seed{SEED}/last.pt",
        f"**/*hstu*{variant}*seed{SEED}*/best.pt",
        f"**/*hstu*{variant}*seed{SEED}*/latest.pt",
        f"**/*hstu*{variant}*seed{SEED}*/last.pt",
    ]
    seen = {str(p) for p in candidates}
    for pattern in patterns:
        for p in mydrive.glob(pattern):
            if str(p) not in seen:
                seen.add(str(p)); candidates.append(p)

    inspected = []
    for p in candidates:
        try:
            ck = torch.load(p, map_location="cpu", weights_only=False)
            if _checkpoint_variant(p, ck) != variant:
                continue
            q = _checkpoint_quality(ck)
            preference = {"best.pt": 2, "latest.pt": 1, "last.pt": 0}.get(p.name, -1)
            inspected.append((q, preference, p))
        except Exception as e:
            print("Checkpoint inspect warning:", p, repr(e))

    if not inspected:
        raise FileNotFoundError(
            f"No usable HSTU {variant} seed-{SEED} checkpoint found under {mydrive}. "
            f"Expected folder was {expected}."
        )

    inspected.sort(key=lambda x: (x[0], x[1]), reverse=True)
    q, _, path = inspected[0]
    RESOLVED_CHECKPOINTS[variant] = path
    print("RESOLVED CHECKPOINT:", {
        "variant": variant,
        "path": str(path),
        "saved_NDCG@10": None if q == float("-inf") else q,
        "alternatives": [str(x[2]) for x in inspected[1:]],
    })
    return path

def checkpoint_path(variant):
    return resolve_checkpoint(variant)

def load_canonical_model(variant, max_item_id):
    path = resolve_checkpoint(variant)
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    cfg = ck["config"]
    n = int(cfg["max_len"] + cfg["max_output_len"])
    g = geometry(variant)
    model = HSTU(
        max_item_id=max_item_id,
        n=n,
        dropout=float(cfg["dropout"]),
        l2_eps=float(cfg.get("l2_eps", 1e-6)),
        **g,
    ).to(DEVICE)
    model.load_state_dict(ck["model"])
    model.eval()
    metrics = ck.get("metrics", {})
    print("CHECKPOINT", variant, path, "epoch", ck.get("epoch"), metrics)
    return model, ck

# Resolve both BEFORE running anything expensive.
for v in VARIANTS:
    print(v, "->", resolve_checkpoint(v))

In [ ]:
# 3) Run the canonical stamp suite.
stamp, summary_df, latency_df = finalize()
display(summary_df)
display(latency_df)
print("HSTU_CANONICAL_v1:", stamp["status"])